![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 22 -- Lab 2: Satellite Scenes with a CNN

An Olympiad-style image task: 5,631 satellite pictures of four kinds of ground (cloudy, desert, green area, water) from Kaggle's Satellite Image Classification dataset, scored by accuracy. The pictures come in two sizes, so you resize them to 64 x 64, split them into training, validation and test parts, write the deck's CNN as a class, write the training loop with a validation pass and a checkpoint, test the best epoch once, and write a submission file. Friday looks like this without the scaffolding.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `mahmoudreda55/satellite-image-classification` (about 23 MB), downloaded by `kagglehub` in the setup cell: a `data` folder with one subfolder per class (1,500 cloudy, 1,131 desert, 1,500 green_area, 1,500 water), loaded with `torchvision.datasets.ImageFolder`. Turn on a GPU in Colab (Runtime, Change runtime type, T4) if you can; on a CPU an epoch takes about half a minute.

---
# Setup

In [ ]:
# kagglehub, torch, torchvision, matplotlib, pandas and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q kagglehub torch torchvision matplotlib pandas scikit-learn

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("mahmoudreda55/satellite-image-classification")
root = os.path.join(path, "data")
tf = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])   # one size for every picture, values 0..1
full = datasets.ImageFolder(root, transform=tf)
CLASSES = full.classes
print(len(full), CLASSES)

---
# Part 1 -- Look and split

The *Task 1: Image Classification* slide: a folder per class is the data, accuracy is the score.

## Task 1: Look at the data

Print how many pictures each class has (`np.bincount(full.targets)` next to `CLASSES`). Print the raw size of `full.samples[0][0]` and of `full.samples[5000][0]` with `Image.open(...).size`: two different sizes, which is why the transform resizes. Then show 8 pictures `full[i]` for `i in range(0, 5631, 700)` with their class names (`permute(1, 2, 0)` for `imshow`).

In [ ]:
# Your code here


## Task 2: Split and load

Split `full` into 3,941 training, 845 validation and 845 test pictures with `random_split(full, [3941, 845, 845], generator=torch.Generator().manual_seed(0))` as `train_set, val_set, test_set` (70 / 15 / 15 percent). Build `train_loader` (batch 64, `shuffle=True`), `val_loader` and `test_loader` (batch 256, no shuffle). Take one batch `xb, yb = next(iter(train_loader))` and print its shapes: `(64, 3, 64, 64)` and `(64,)`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_set) == 3941 and len(test_set) == 845 and tuple(xb.shape) == (64, 3, 64, 64), 'split: random_split into 3941 / 845 / 845 with a seeded generator; batches of 64'
print('Task 2 passed')

---
# Part 2 -- The model as a class

The *In Code: The Model as a Class* slide. At 64 x 64, three blocks leave a (64, 8, 8) map.

## Task 3: Write the model class

Write `class SceneCNN(nn.Module)` with `__init__(self, n_classes=4)`: `super().__init__()`, then `self.features = nn.Sequential(...)` of three blocks `nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)` for channels 3 to 16, 16 to 32, 32 to 64, and `self.classifier = nn.Linear(64 * 8 * 8, n_classes)`; and `forward(self, x)` that runs `self.features`, flattens with `x.flatten(1)` and returns `self.classifier(x)`. Create `model = SceneCNN().to(device)`, print `model(xb.to(device)).shape` (expected `(64, 4)`) and the parameter count `n_params`.

Syntax hint (the shape of the call, not the answer):

```python
class MyNet(nn.Module):
    def __init__(self, ...):
        super().__init__()
        self.features = nn.Sequential(...)
        self.classifier = nn.Linear(...)
    def forward(self, x):
        ...
        return scores
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(model(xb.to(device)).shape) == (64, 4) and n_params == 448 + 4640 + 18496 + 64 * 8 * 8 * 4 + 4, 'SceneCNN: three blocks 3->16->32->64, padding 1, MaxPool2d(2), then Linear(4096, 4)'
print('Task 3 passed')

## Task 4: The untrained network on the validation set

Create `loss_fn = nn.CrossEntropyLoss()`. With `model.eval()` and inside `torch.no_grad()`, loop over `val_loader`, move each batch to `device`, add `loss_fn(logits, yb).item() * len(yb)` to a running total and the number of correct `argmax` predictions to another, then divide both by `len(val_set)` as `val_loss0, val_acc0` and print them: about `ln(4) = 1.386` and about 0.25 (four classes, random weights).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert 1.2 < val_loss0 < 1.6 and val_acc0 < 0.6, 'untrained: mean cross-entropy near ln(4), accuracy near chance'
print('Task 4 passed')

---
# Part 3 -- The training loop

The *Same Training Loop* slide: Day 16's loop, written out, with the validation pass inside and the best epoch saved.

## Task 5: Write the loop

Set `EPOCHS = 8`, `opt = torch.optim.Adam(model.parameters(), lr=1e-3)`, `history = []`, `best_acc = 0.0`. For each epoch: `model.train()`, loop over `train_loader` with the four moves (move the batch to `device`; `loss = loss_fn(model(xb), yb)`; `opt.zero_grad()`; `loss.backward()`; `opt.step()`), summing `loss.item() * len(yb)` into `train_loss`; then `model.eval()` and, inside `torch.no_grad()`, the validation pass of Task 4 giving `val_loss` and `val_acc`; divide the sums by the set sizes; append `(train_loss, val_loss, val_acc)` to `history`; if `val_acc > best_acc`, update `best_acc` and `torch.save(model.state_dict(), "best_scene.pt")`; print the epoch's three numbers. Expected: validation accuracy above 0.9 within a few epochs.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(history) == 8 and best_acc > 0.85 and os.path.exists('best_scene.pt'), 'loop: 8 epochs, the four moves per batch, a validation pass per epoch, the checkpoint saved when val_acc improves'
print('Task 5 passed')

## Task 6: Read the curves

Plot the training loss and the validation loss against the epoch on the left, and the validation accuracy on the right (two subplots, markers, labels, titles, a legend). Which epoch did the checkpoint keep, and are the two loss curves still together?

In [ ]:
# Your code here


---
# Part 4 -- Test once, then submit

An Olympiad hand-in: the checkpoint with the best validation score, scored once on pictures never used for any decision, then the file.

## Task 7: Test the best checkpoint

Create `best = SceneCNN().to(device)`, load the checkpoint with `best.load_state_dict(torch.load("best_scene.pt", map_location=device))`, set `best.eval()`, and inside `torch.no_grad()` loop over `test_loader` collecting `best(xb).argmax(1).cpu()` into a list and `yb` into another. Concatenate them with `torch.cat(...).numpy()` as `pred_test` and `y_test`, print the test accuracy `(pred_test == y_test).mean()`, and print `confusion_matrix(y_test, pred_test)` as a DataFrame with the class names on both axes.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pred_test.shape == (845,) and test_acc > 0.85, 'test: load_state_dict from best_scene.pt, predict test_loader in order, one label per picture'
print('Task 7 passed')

## Task 8: Look at the mistakes

Find the positions where `pred_test != y_test` and show up to 8 of those test pictures (`test_set[i][0].permute(1, 2, 0)`) with the title `predicted / true` using the class names. What do the confused pairs have in common?

In [ ]:
# Your code here


## Task 9: Write the submission

Save `pd.DataFrame({"id": np.arange(len(test_set)), "label": [CLASSES[p] for p in pred_test]})` to `submission.csv` with `index=False`, read it back, and print its shape and first 5 rows. The file must have exactly 845 rows and only the four class names as labels.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert check.shape == (845, 2) and set(check['label'].unique()) <= set(CLASSES), 'submission.csv: 845 rows, columns id and label, labels are class names'
print('Task 9 passed')

---
## Task 10: Reflect

The pictures came in two sizes and the transform resized them all to 64 x 64. Why must every picture in a batch have the same shape, and what else does the resize throw away? The checkpoint was chosen on the validation pictures and the test pictures were scored once. Why not choose the epoch on the test score, and what would the number mean if you did? The loop you wrote is the same as Day 16's with a different model class. Name the one place in the notebook you would change to try a wider network, and the one number you would watch to decide whether to keep it.

*Your answers here*

---
## Conclusion

- **Kaggle images in PyTorch**: `kagglehub` downloaded the folder, `ImageFolder` turned folder names into labels, a `Resize` transform gave every picture one shape, `random_split` made training, validation and test parts.
- **The model as a class**: `SceneCNN` with `__init__` (three conv blocks and a `Linear`) and `forward`, checked by its output shape and parameter count before training, with an untrained loss of ln(4).
- **The loop, written out**: the four moves per batch, a validation pass per epoch, the best epoch saved; the curves told you which epoch and whether the losses stayed together.
- **The hand-in**: the checkpoint scored once on the test part, a confusion matrix and the wrong pictures, and a checked submission file.

---

Made By **Sattam Altwaim**